In [1]:
# Import necessary libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType, IntegerType

# Initialize Spark session
spark = SparkSession.builder \
    .appName("UsingUDFsDemo") \
    .getOrCreate()

In [2]:
# Create a sample DataFrame
data = [
    (1, "John", 28),
    (2, "Emily", 22),
    (3, "Michael", 35),
    (4, "Sarah", 30),
    (5, "David", 40),
    (6, "Alice", 29)
]

columns = ["id", "name", "age"]

df = spark.createDataFrame(data, columns)

print("Original DataFrame:")
df.show()

Original DataFrame:
+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|   John| 28|
|  2|  Emily| 22|
|  3|Michael| 35|
|  4|  Sarah| 30|
|  5|  David| 40|
|  6|  Alice| 29|
+---+-------+---+



In [3]:
# Writing a UDF for Age Category
def age_category(age):
    if age < 30:
        return "Under 30"
    elif 30 <= age < 40:
        return "30s"
    else:
        return "40+"

In [4]:
# Register the UDF with a return type of String
age_category_udf = udf(age_category, StringType())

In [5]:
# Apply the UDF to the DataFrame
df_with_age_category = df.withColumn("age_category", age_category_udf(df["age"]))

In [6]:
print("DataFrame with Age Category (Using UDF):")
df_with_age_category.show()

DataFrame with Age Category (Using UDF):
+---+-------+---+------------+
| id|   name|age|age_category|
+---+-------+---+------------+
|  1|   John| 28|    Under 30|
|  2|  Emily| 22|    Under 30|
|  3|Michael| 35|         30s|
|  4|  Sarah| 30|         30s|
|  5|  David| 40|         40+|
|  6|  Alice| 29|    Under 30|
+---+-------+---+------------+



In [7]:
# Registering the UDF for Reuse
spark.udf.register("age_category_udf", age_category, StringType())

<function __main__.age_category(age)>

In [8]:
# Use the UDF in SQL queries
df_with_age_category.createOrReplaceTempView("people")

print("Using UDF in SQL Query:")
df_sql_result = spark.sql("SELECT id, name, age, age_category_udf(age) AS age_category FROM people")
df_sql_result.show()

Using UDF in SQL Query:
+---+-------+---+------------+
| id|   name|age|age_category|
+---+-------+---+------------+
|  1|   John| 28|    Under 30|
|  2|  Emily| 22|    Under 30|
|  3|Michael| 35|         30s|
|  4|  Sarah| 30|         30s|
|  5|  David| 40|         40+|
|  6|  Alice| 29|    Under 30|
+---+-------+---+------------+

